# E9 Multistage Training

Author: Arush Arora

## Introduction
This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the graph before expecting it to serve the LLM with **multiplicative** GREPs, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

## The R-PEARL GNN
The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

### Graph Convolutional Network (GNN)
The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

#### Transformer
The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$X = \tilde{X} + P$$

$${Z}_{1:t}^{(L)} = \operatorname{Trf}\bigg({X}_{1:t}, {\mathcal{T}}_l\bigg) \qquad {\mathcal{T}}_l = \begin{bmatrix}
{Q}_l & {K}_l & {V}_l & \left({W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big({Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

## Graph-Augmented LLM
The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$ $$\utilde{P} = \hat{\mathbb{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \utilde{S}, \mathcal{H}\Big)$$

$$\utilde{X} = \utilde{\tilde{X}} + \utilde{P}$$

$${\utilde{Z}}_{1:t}^{(L)} = \operatorname{Trf}\bigg({\utilde{X}}_{1:t}\bigg)$$

In [1]:
%env CUDA_VISIBLE_DEVICES=1

env: CUDA_VISIBLE_DEVICES=1


In [2]:
# Import modules.
import gc
import torch
import random
import numpy as np
import sympy as sp
from torch import nn
import networkx as nx

from prism.models import inference, gnn_llm, gt, loaders
from prism.eval import evaluate, loading
from prism.data import data

In [3]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 0, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [4]:
# Standard options.
checkpoint = '../outputs/e8_new_base_models/e8_graph_mask_llm_gemma-4-12b-it_r16_4bit_v40uxzgf/'
eval_path = '../data_store/revised/gen/nav100_n30_gemma_data/split/test_graphs'
include_edge_list = False
use_pretrained = True
device = 'cuda:0'

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Setup the Gemma 4 model from Hugging Face.
llm = AutoModelForCausalLM.from_pretrained("google/gemma-4-12B-it", dtype="auto", device_map="auto")

# Initialize a barebones/pretrained planner for testing.
if use_pretrained:
    model = gnn_llm.GraphMaskLLM(llm, use_edges=True)
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-12B-it")
else:
    model, tokenizer = loaders.graph_augmented_llm_from_pretrained(
        checkpoint, load_in_4bit=True, device=device,
    )

planner = inference.GraphAugmentedInMemoryLLM(model, tokenizer, include_edge_list)

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [ ]:
from datasets import load_dataset

# Load in training dataset.
full_dataset = load_dataset("json", data_files=["../data/gen/nav100_n10_gemma_data/split/formatted_all_new_2turn__train.json"], split="train")
full_dataset = data.preprocess_dataset(
    full_dataset, tokenizer,
    architecture="graph_mask_llm",
    text_edge_list=include_edge_list,
)

In [ ]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = loading.load_samples_by_graph(eval_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [eval_data[random.randint(0, len(eval_data) - 1)]]}

In [8]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_009.html


## Experiments

### §1 Testing a Pretrained `GraphMaskLLM`
The goal here is to test the model without any modifications to its internal architecture. By retrieving the necessary tensors for comparison, we can compute the perturbation of the attention logits and softmax distributions of the following matrix with $\mathbf{M}$ representing the block-matrix containing the graph adjacency $\mathbf{A} = \Big[\mathbb{I}\big((u, v) \in \mathcal{E}\big)\Big]_{u, v \in \mathcal{V}}$:
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1}\right) \odot M\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1}\right) \odot M\right]}\right]^\top_{t \in [N]}$$

In [9]:
"""
results = evaluate.eval_model_multiple_graphs(
    model, tokenizer, eval_data,
    include_edge_list=include_edge_list,
    use_icl=False,
    permutation=None,
    on_graph_done=None
)
results[graph_file].path_metrics
"""

'\nresults = evaluate.eval_model_multiple_graphs(\n    model, tokenizer, eval_data,\n    include_edge_list=include_edge_list,\n    use_icl=False,\n    permutation=None,\n    on_graph_done=None\n)\nresults[graph_file].path_metrics\n'

### §2 Testing a Pretrained `GraphMaskLLM` with PE Injection into M (No Fintuning)
We would now like to instantiate a Graph Transformer and train it to replicate the graph adjacency matrix $\mathbf{A}$, defined above, to see if we achieve similar results.

In [10]:
# Instantiate a Graph Transformer.
gt = gt.GraphTransformer(
    num_layers=3,
    pe_hidden_channels=256,
    pe_num_layers=2,
    d_model=1024,
    heads=8,
    num_samples=40,
    dropout=0.1,
    k_pe=2,
    k_gt=2,
    eps=1e-6,
    use_layer_norm=True
)

#### Numeric Visualizations with SymPy
Using the `render_matrix()` function defined at the very beginning of this notebook, we first explore the procedure needed to preprocess a pre-training set for the Graph Transformer to reconstruct the eigenbasis of the graph adjacency given a scene graph PyTorch `Data` object.

In [11]:
# Prepare a graph from the data to be used in the Graph Transformer.
from torch_geometric.utils import to_dense_adj
from prism.data import utils
import numpy as np
import sympy as sp

graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
adj_list = model._node_adjacency(graph, device=device).to(torch.float32)
adj = to_dense_adj(graph.edge_index).squeeze().cuda()

# Take the difference between the _node_adjacency() matrix and the to_dense_adj() matrix (for debugging).
render_matrix(adj_list.int() - adj)

Matrix([
[1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0

In [12]:
_, eigh_vec = torch.linalg.eigh(adj_list)
render_matrix(eigh_vec, sig_figs=3)

Matrix([
[ 0.0732, -0.0767,   0.102,   0.0136, -0.0852,   0.108,  -0.212,  -0.0243,    -0.09,  -0.233,  0.000291,    0.014,   0.0851,   0.328,  -0.445,  -0.136,   -0.401,  0.0266,    0.361,  -0.216,  -0.0294, -0.0303, -0.0809, -0.0432,    0.398, -0.0304,    0.072, -0.0391,  0.0732, -0.0258],
[-0.0535,  0.0648, -0.0711,  -0.0422,   0.114,  0.0247,    0.13,    0.121,    0.016,  -0.417,     0.174,   -0.063,    0.204,   0.277,   0.129,  -0.277,    0.171,   0.179,   -0.455,  -0.232,   0.0937,   0.198,    0.16,  0.0901,    0.216,   0.223,   -0.131, -0.0157,  0.0499, -0.0168],
[ 0.0188,  -0.112,  0.0377,    0.182,  0.0381,  -0.038,   0.107,  -0.0748,   -0.109,   0.193,     0.174,    -0.22,   0.0348,   0.322,   0.407,  -0.159,   -0.115,   0.305,    0.415,   0.103,    0.353,  0.0356, -0.0822, -0.0584,   -0.168,  0.0767,   -0.209, -0.0122,  0.0682, -0.0295],
[  0.038, -0.0171,  -0.127,   0.0194,  -0.131,   0.109,  0.0242, -0.00372,    0.116,  0.0212,    0.0742,    -0.58,    0.215,  -0.117,  -0.1

In [13]:
# Feed the matrix to the Graph Transformer.
out = gt(graph).to(device)
out = out @ out.T / torch.linalg.matrix_norm(out)
render_matrix(out, 3)

Matrix([
[6.27, 5.79, 5.58, 5.42, 5.71, 5.63,  5.6, 5.39, 5.61, 5.54, 5.55, 5.52, 5.52, 5.43, 5.41, 5.55, 5.44, 5.53,  5.6,  5.5,  5.6,  5.4, 5.59,  5.7, 5.62,  5.5,  5.6, 5.59, 5.63, 5.75],
[5.79, 6.55, 5.74, 5.71, 5.82, 5.77, 5.73, 5.75, 5.89, 5.76, 5.74, 5.82, 5.84, 5.74, 5.77,  5.8, 5.79, 5.89, 5.89,  5.8, 5.81, 5.69, 5.84,  5.7, 5.74, 5.65, 5.85, 5.84, 5.81, 5.86],
[5.58, 5.74, 6.12, 5.48, 5.62, 5.57, 5.51, 5.49, 5.59, 5.48,  5.5, 5.52, 5.54, 5.41, 5.42, 5.47, 5.43, 5.52, 5.54, 5.42,  5.5, 5.34, 5.52, 5.57, 5.52,  5.4,  5.6,  5.5, 5.57, 5.59],
[5.42, 5.71, 5.48, 6.24, 5.55, 5.58,  5.5, 5.68,  5.6, 5.45, 5.54, 5.61, 5.65, 5.59,  5.7, 5.61, 5.67, 5.68,  5.7, 5.63, 5.62,  5.5, 5.69, 5.39, 5.41, 5.37, 5.59, 5.56, 5.53, 5.47],
[5.71, 5.82, 5.62, 5.55, 6.38, 5.77, 5.57, 5.54, 5.68, 5.56, 5.59, 5.62, 5.65, 5.55, 5.59, 5.58, 5.56, 5.64, 5.61, 5.56, 5.65, 5.42, 5.69, 5.69, 5.67, 5.49, 5.73, 5.67, 5.72, 5.77],
[5.63, 5.77, 5.57, 5.58, 5.77, 6.26, 5.53, 5.58, 5.67, 5.47, 5.58, 5.61, 5.69, 5.

In [14]:
# Compute the loss.
loss_fn = nn.MSELoss()
loss_fn(eigh_vec, out).squeeze()

tensor(32.9690, device='cuda:0', grad_fn=<SqueezeBackward0>)

#### Pre-Training of Graph Transformer on Eigenbases
Next, we actually preprocess and train the Graph Transformer using the steps defined above.

In [15]:
# Import Modules.
from torch_geometric.loader import DataLoader

# Configure the training and test datasets.
train_dataset, _ = loading.load_samples_by_graph(
    '../data_store/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)

# Add eigenbasis training outputs for reconstruction.
graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in train_dataset.items()]
adjs = [model._node_adjacency(graph, device=device).to(torch.float32) for graph in graphs]
eigh_vecs = [torch.linalg.eigh(adj)[1] for adj in adjs]

for i, graph in enumerate(graphs):
    graph.y = eigh_vecs[i]
    graph.x.to(device)
    graph.edge_index.to(device)

In [17]:
# Train the Graph Transformer to reconstruct the eigenvectors of the graph adjacency.
from torch.optim.lr_scheduler import StepLR
from torch_geometric.loader import DataLoader

batch_size = 20
epochs = 100

def train_loop(dataloader, model, loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(dataloader.dataset)
    model.to(device)
    model.train()
    for i in range(epochs):
        print(f"=============\nEpoch #{i}\n=============")
        for j, graph in enumerate(dataloader.dataset):
            # Compute prediction and loss.
            pred = model(graph)
            pred = pred @ pred.T / torch.linalg.matrix_norm(pred)
            loss = loss_fn(pred.to(device), graph.y)

            # Backpropagation.
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            if scheduler:
                scheduler.step()

            # Results.
            if j % batch_size == 0:
                loss, current = loss.item(), j
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


loss_fn = nn.MSELoss()
data = DataLoader(graphs, batch_size=5)
optimizer = torch.optim.AdamW(gt.parameters(), lr=3e-4)
# scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
train_loop(data, gt, loss_fn, optimizer, None, batch_size=batch_size, epochs=epochs)

Epoch #0
Loss: 29.919165  [    0/   40]
Loss: 0.378381  [   20/   40]
Epoch #1
Loss: 0.278973  [    0/   40]
Loss: 0.220439  [   20/   40]
Epoch #2
Loss: 0.210384  [    0/   40]
Loss: 0.182745  [   20/   40]
Epoch #3
Loss: 0.185885  [    0/   40]
Loss: 0.168218  [   20/   40]
Epoch #4
Loss: 0.182271  [    0/   40]
Loss: 0.161877  [   20/   40]
Epoch #5
Loss: 0.173762  [    0/   40]
Loss: 0.158470  [   20/   40]
Epoch #6
Loss: 0.172184  [    0/   40]
Loss: 0.155337  [   20/   40]
Epoch #7
Loss: 0.166021  [    0/   40]
Loss: 0.151805  [   20/   40]
Epoch #8
Loss: 0.164402  [    0/   40]
Loss: 0.149086  [   20/   40]
Epoch #9
Loss: 0.162082  [    0/   40]
Loss: 0.145930  [   20/   40]
Epoch #10
Loss: 0.156973  [    0/   40]
Loss: 0.143785  [   20/   40]
Epoch #11
Loss: 0.159688  [    0/   40]
Loss: 0.143882  [   20/   40]
Epoch #12
Loss: 0.160059  [    0/   40]
Loss: 0.142398  [   20/   40]
Epoch #13
Loss: 0.153701  [    0/   40]
Loss: 0.139079  [   20/   40]
Epoch #14
Loss: 0.152737  [  

In [18]:
torch.save(gt.state_dict(), '../outputs/e9_multistage_training/eigenbasis_gt_normed.pt')
# gt.load_state_dict(torch.load('../outputs/e9_multistage_training/eigenbasis_gt.pt'))

#### Evaluation of Pre-Trained GT on Eigenbases
We now test the trained model on the evaluation dataset. First, we we will render the matrices to see their differences visually.

In [19]:
# Load in training data.
test_dataset, _ = loading.load_samples_by_graph(
    '../data_store/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)

# Add eigenbasis test outputs for reconstruction.
test_graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for k, v in train_dataset.items()]
test_adjs = [model._node_adjacency(graph, device=device).to(torch.float32) for graph in graphs]
test_eigh_vecs = [torch.linalg.eigh(adj)[1] for adj in adjs]

for i, test_graph in enumerate(test_graphs):
    test_graph.y = eigh_vecs[i]
    test_graph.x.to(device)
    test_graph.edge_index.to(device)

In [20]:
# Prepare a graph from the data to be used in the Graph Transformer.
graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
adj_list = model._node_adjacency(graph, device=device).to(torch.float32)
_, eigh_vec = torch.linalg.eigh(adj_list)
render_matrix(eigh_vec, sig_figs=3)

Matrix([
[ 0.0732, -0.0767,   0.102,   0.0136, -0.0852,   0.108,  -0.212,  -0.0243,    -0.09,  -0.233,  0.000291,    0.014,   0.0851,   0.328,  -0.445,  -0.136,   -0.401,  0.0266,    0.361,  -0.216,  -0.0294, -0.0303, -0.0809, -0.0432,    0.398, -0.0304,    0.072, -0.0391,  0.0732, -0.0258],
[-0.0535,  0.0648, -0.0711,  -0.0422,   0.114,  0.0247,    0.13,    0.121,    0.016,  -0.417,     0.174,   -0.063,    0.204,   0.277,   0.129,  -0.277,    0.171,   0.179,   -0.455,  -0.232,   0.0937,   0.198,    0.16,  0.0901,    0.216,   0.223,   -0.131, -0.0157,  0.0499, -0.0168],
[ 0.0188,  -0.112,  0.0377,    0.182,  0.0381,  -0.038,   0.107,  -0.0748,   -0.109,   0.193,     0.174,    -0.22,   0.0348,   0.322,   0.407,  -0.159,   -0.115,   0.305,    0.415,   0.103,    0.353,  0.0356, -0.0822, -0.0584,   -0.168,  0.0767,   -0.209, -0.0122,  0.0682, -0.0295],
[  0.038, -0.0171,  -0.127,   0.0194,  -0.131,   0.109,  0.0242, -0.00372,    0.116,  0.0212,    0.0742,    -0.58,    0.215,  -0.117,  -0.1

In [21]:
# Feed the matrix to the Graph Transformer.
out = gt(graph).to(device)
out = out @ out.T / torch.linalg.matrix_norm(out)
render_matrix(out, 3)

Matrix([
[    0.41,    0.0143, 0.00346,    0.0396,   0.00794,  0.0155,   0.0861,    0.058,  0.00433, -0.00756,   0.0413,   0.0495,   0.0097,   0.0416, -0.00419,    0.0216,  0.00713,   0.0318,   0.0414,   0.0299,   0.0103, -0.00689,  0.0122,  0.00815,   0.0561,   0.0437,    0.0488,   0.0247,  -0.00849,    0.0439],
[  0.0143,     0.396, 0.00802,    0.0574,   0.00719,  0.0293,   0.0257,  0.00447,   0.0476,   0.0451,     0.09,  0.00839,   0.0221,   0.0442,   0.0438,    0.0167,   0.0404,   0.0291,   0.0318,   0.0299,    0.028,   0.0254,  0.0208,   0.0171,   0.0107,   0.0518,    0.0111,   0.0249, -0.000908, -0.000753],
[ 0.00346,   0.00802,   0.265,    0.0434,    0.0285,  0.0435,   0.0501,   0.0281,   0.0319,   0.0301,   0.0237,   0.0165,   0.0511,   0.0413,   0.0437,    0.0194,   0.0345,  0.00462,   0.0453,  0.00145,   0.0318,   0.0581,  0.0575,   0.0449,   0.0274,   0.0317,    0.0447,   0.0182,     0.052,    0.0404],
[  0.0396,    0.0574,  0.0434,     0.621, -0.000933,  0.0635,   0.0334,  

In [ ]:
# Evaluate the Graph Transformer on its reconstruction of eigenvectors of test graph adjacencies.
def test_loop(dataloader, model, loss_fn):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for graph in dataloader.dataset:
            pred = model(graph)
            pred = pred @ pred.T / torch.linalg.matrix_norm(pred)
            test_loss += loss_fn(pred.to(device), graph.y).item()
            correct += torch.allclose(pred, graph.y)

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

del train_dataset, data, graphs, adjs, eigh_vecs
gc.collect()
data = DataLoader(test_graphs, batch_size=5)
test_loop(data, gt, loss_fn)

Test Error: 
 Accuracy: 0.0%, Avg loss: 0.201223 



#### Performing the Injection with the Evaluated Pre-Trained GT on Eigenbases
We now subclass the `GraphMaskLLM` class to replicate the adjacency matrix using our trained Graph Transformer instead of through the original `_node_adjacency()` method defined in the superclass. Such a system will serve as a proof of concept for the next experiment.

In [28]:
# Define and instantiate the PEInjectionGraphMaskLLM class.
from prism.models.gnn_llm import GraphMaskLLM


class PEInjectionGraphMaskLLM(GraphMaskLLM):
    def build_structural_mask(self, seq_len, graphs, injection_maps, device, dtype=None):
        """Additive attention bias ``[B, 1, seq, seq]`` — 0 allowed, ``finfo.min`` blocked.

        ``bias[b,0,i,j] = finfo.min`` iff tokens i and j BOTH belong to graph nodes
        AND those nodes are non-adjacent (within ``k_hops``). Every other entry
        (node↔non-node, non-node↔non-node, same node, adjacent) stays 0. Because it
        is ADDED to the model's causal/sliding mask, blocking only ever removes
        already-causal pairs. Each node-token row keeps BOS (a non-node) and its own
        diagonal, so no row is fully masked (no softmax NaN).
        """
        if dtype is None:
            dtype = self.llm.get_input_embeddings().weight.dtype
        B = len(injection_maps)
        neg = torch.finfo(dtype).min
        bias = torch.zeros(B, 1, seq_len, seq_len, device=device, dtype=dtype)
        for b in range(B):
            g = graphs[b]
            # token position -> node id (-1 for non-node tokens). Spans are disjoint
            # (build_injection_map dedups longest-first), so each token maps to one node.
            tok2node = torch.full((seq_len,), -1, dtype=torch.long, device=device)
            for node_idx, spans in injection_maps[b].items():
                for start, end in spans:
                    end = min(end, seq_len)
                    if start < end:
                        tok2node[start:end] = node_idx
            node_pos = (tok2node >= 0).nonzero(as_tuple=True)[0]
            if node_pos.numel() == 0:
                continue
            eigh_vecs = gt(g)
            adj = eigh_vecs @ eigh_vecs.T
            nid = tok2node[node_pos]
            allowed = adj[nid][:, nid]
            blocked = ~allowed
            if blocked.any():
                bi, bj = blocked.nonzero(as_tuple=True)
                bias[b, 0, node_pos[bi], node_pos[bj]] = neg
        return bias

In [ ]:
results = evaluate.eval_model_multiple_graphs(
    model, tokenizer, eval_data,
    include_edge_list=include_edge_list,
    use_icl=False,
    permutation=None,
    on_graph_done=None
)
results[graph_file].path_metrics